# 7 Your own series, your own negative result

Notebooks 01-06 ran the toolkit on series somebody else picked: daily sunspot counts,
solar power, Australian electricity demand. This notebook is where you pick your
own, from the same archive, and run the whole lesson's toolkit on it yourself:

1. establish the naive/seasonal-naive/linear baseline (`02_basemodels`' habit) --
   **before** you touch anything else;
2. decompose it and try N-BEATS (`03_decomposition_and_nbeats`);
3. decide, and defend, whether probabilistic forecasting is worth the extra
   machinery on this series (`04_probabilistic`);
4. try one gated architecture (`05_gates_highways_moe`) and measure it against a
   capacity-matched plain control, not against nothing.

**The deliverable is a short report that must contain at least one negative result you
produced and are willing to stand behind** — same requirement as
`1_pytorch_intro/05_exercises`. "We tried it, the baseline won, here is the number" is
a complete and respectable finding. A report that only shows things working has not
done the assignment.

The `Monash-University/monash_tsf` configs used so far do not repeat: `sunspot`,
`solar_10_minutes`, `australian_electricity_demand`. Pick a different one —
`traffic_hourly`, `nn5_daily`, `kaggle_web_traffic_weekly`, `covid_deaths`,
`us_births`, `saugeenday` and more all load the same way (§7.1 below is the pattern,
worked on `traffic_hourly` so you have a template that runs end to end).

## 7.1 The pattern, worked once

`traffic_hourly` — San Francisco Bay Area road occupancy, hourly, 862 sensors, values
in [0, 1]. Genuinely noisier than anything else in this lesson (main_principles.md
§3.8 measured its spectral top-5/total at 0.38, the lowest of every series checked) —
picked deliberately so the worked example is not another easy win.


In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from ett import window_index
from notebooktester import param

W, H = 168, 24
EPOCHS = param(10, test=1)
SEEDS = (0, 1, 2)


def load_series(config: str, n_series: int = 4, min_len: int = 6 * (W + H)) -> list[torch.Tensor]:
    # the same loader shape 03_decomposition_and_nbeats and 06_search both use --
    # swap `config` for any Monash-University/monash_tsf name and nothing else changes.
    ds = load_dataset(
        "Monash-University/monash_tsf", revision="refs/convert/parquet",
        data_files={"test": f"{config}/test/0000.parquet"},
    )["test"]
    out = []
    for row in ds:
        s = np.asarray(row["target"], dtype="float32")
        s = s[~np.isnan(s)]
        if len(s) >= min_len:
            out.append(torch.tensor(s))
        if len(out) >= n_series:
            break
    return out


def split_scale_window(series: list[torch.Tensor], stride: int = 6) -> dict:
    parts = {"train": [[], []], "val": [[], []], "test": [[], []]}
    for s in series:
        n = len(s)
        cuts = {"train": (0, int(0.6 * n)), "val": (int(0.6 * n), int(0.8 * n)),
                "test": (int(0.8 * n), n)}
        mu = s[: cuts["train"][1]].mean()
        sigma = s[: cuts["train"][1]].std().clamp(min=1e-6)
        for split, (a, b) in cuts.items():
            v = (s[a:b] - mu) / sigma
            win = v[window_index(len(v), W + H, stride)]
            parts[split][0].append(win[:, :W])
            parts[split][1].append(win[:, W:])
    return {k: (torch.cat(v[0]), torch.cat(v[1])) for k, v in parts.items()}


data = split_scale_window(load_series("traffic_hourly"))
xtr, ytr = data["train"]
xva, yva = data["val"]
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")

In [ ]:
def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


mase_scale = float((xtr[:, 1:] - xtr[:, :-1]).abs().mean())   # 02_basemodels 2.2
naive = xva[:, -1:].expand(-1, H)
seasonal = xva[:, -24:][:, :H]

print(f"{'naive (repeat last)':22s} MAE {mae(yva, naive):.4f}   MASE {mae(yva, naive) / mase_scale:.3f}")
print(f"{'seasonal naive (24h)':22s} MAE {mae(yva, seasonal):.4f}   MASE {mae(yva, seasonal) / mase_scale:.3f}")


def fit(make_model, seeds=SEEDS, epochs=EPOCHS, batch_size=64, lr=1e-3):
    scores = []
    for seed in seeds:
        torch.manual_seed(seed)
        model = make_model()
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.MSELoss()
        loader = DataLoader(TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True)
        for _ in range(epochs):
            for xb, yb in loader:
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                opt.step()
        model.eval()
        with torch.no_grad():
            scores.append(mae(yva, model(xva)))
    return float(np.mean(scores)), float(np.std(scores))


class Linear(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(W, H)

    def forward(self, x):
        return self.fc(x)


lin_mae, lin_std = fit(Linear)
print(f"{'linear':22s} MAE {lin_mae:.4f} +/- {lin_std:.4f}   MASE {lin_mae / mase_scale:.3f}")

**Write down what you see before moving on.** On `traffic_hourly`, is the naive or the
seasonal-naive baseline stronger? Does the trained linear layer clear either of them by
more than its own seed spread? That answer is the first line of your report, and it
should be a number, not an impression — `02_basemodels` §2.1-2.2 is the reasoning to
reuse, not just the code.


## 7.2 Decompose it, try N-BEATS

`03_decomposition_and_nbeats` §3.2-3.5 has every class you need (`TrendBasis`,
`SeasonalityBasis`, `NBeatsBlock`, `NBeats`, `interpretable()`, `generic()`) — copy
them, do **not** re-derive them, and point `period=` at whatever `traffic_hourly`'s
spectrum (or your own series') actually shows, the way §3.1 read solar power's off an
FFT before assuming 24 was right.

Run the same three-way baseline table §3.4 does (linear / N-BEATS generic / N-BEATS
interpretable). Does the constrained architecture earn anything here, or does the
linear layer win again? Either answer is fine — §3.6 measured both directions on two
different series and that was the point, not an accident.

## 7.3 Is probabilistic forecasting worth it here?

Not "run `04_probabilistic`'s code and report a CRPS" — first make the case in words.
§4.1's neighbour-search trick (plot 40 near-identical histories, look at how different
the outcomes were) is cheap and answers the actual question: **is there aleatoric
uncertainty here worth modelling**, or would a point forecast already say most of what
there is to say? If the fan in your plot is narrow, say so and explain why building a
Gaussian or quantile head on top would be effort spent on a question your data barely
asks. If it is wide, build the free baseline from §4.3 and one trained head, and report
CRPS against it, not in isolation.

## 7.4 One gated architecture

`05_gates_highways_moe` §5.2-5.5 has `Gate`, `Highway`, and `MoE` wrapped around a
plain hidden layer, plus the capacity-matched control from §5.6 that makes the
comparison honest. Pick **one** mechanism, build it on your series' window, and compare
it to a plain block with the *same parameter count* — not to the naive baseline, which
a gate was never competing with in the first place.

## Deliverable

A short write-up (or a presentation built from mlflow-logged runs, the way
`1_pytorch_intro/05_exercises` closes) covering:

1. Your series (name, config, why you picked it) and the naive/seasonal-naive/linear
   baseline table, **established first**.
2. The decomposition + N-BEATS result, with the period you chose and why.
3. Your probabilistic-forecasting verdict — worth it or not, with the neighbour-fan
   evidence that backs the call either way.
4. Your gated-architecture result against its capacity-matched control.
5. **At least one negative result** — a thing you tried that did not beat its baseline,
   stated as plainly as the things that did. If everything you tried won, you have not
   tried enough things, or you are not reporting honestly. Either is worth noticing.

## Exercises

**E1 — pick deliberately, not randomly.** Before loading anything, read the Monash
config list (`main_principles.md` §3.8 has several already characterised) and predict:
does your chosen series have a clean spectral peak like solar power, or a spread-out
one like traffic? Check your prediction against the data once you have loaded it.

**E2 — the MASE scale is not optional.** Whatever your final number is, report the MASE
denominator it was computed against, in the same sentence. A MASE with no stated
baseline is decoration, not a result — `02_basemodels` §2.2 says why.

**E3 — three seeds, always.** Every trained number in your report should come with a
+/- from at least 3 seeds. A single-seed win is an anecdote until you have checked it
survives re-running.

**E4 — search it.** If you have time, point `06_search`'s `param_space` at your own
series instead of `solar_10_minutes` and see whether the mechanism ranking from §7.4
holds up under a real search rather than one hand-picked config.